# Yelp Imbalanced Split Generator

**Run this notebook before the focal / contrastive training notebooks.**

It produces `yelp_split_imbalanced.csv` — the full unbalanced dataset split for focal-loss and contrastive training.

### How the split is constructed

1. Loads all four Yelp datasets (YelpZip, YelpChi ×2, YelpNYC) with rating + date metadata — ~1,035,045 reviews, no balancing
2. **Removes collection-duplicate reviews** — the same physical review copied across overlapping source files (YelpNYC ⊂ YelpZip). Dedup key = (normalized text, rating, date), identical to `prepare_yelp_split.ipynb`. Natural duplicates (different reviewers writing the same short text like "good") differ in rating/date and are preserved.
3. **Pins** val and test: any review whose normalized text appeared in `yelp_split.csv` val/test is forced into the same split here — guaranteeing zero leakage when evaluating on the `yelp_split.csv` test set
4. The remaining reviews are split to hit 80/10/10 proportions overall at the unique-text-group level (zero leakage)
5. Final val/test = pinned rows + extra rows from the remaining pool

### Why this design matters

`evaluate_yelp_ablation.py` evaluates all models on the **same** `yelp_split.csv` test set.
By pinning those test groups here, the focal/contrastive training data is **guaranteed** to never contain a test-set review.
The leakage check at the end of this notebook verifies this explicitly.

---
### Instructions

**Prerequisites — files that must already be on your Google Drive:**
1. `yelp_split.csv` — generated by `prepare_yelp_split.ipynb` (run that first if not already done)
2. The raw Yelp dataset files in these exact locations (default paths assume `MyDrive/` root):
   - `YelpZip/metadata`
   - `YelpZip/reviewContent`
   - `YelpChi/output_meta_yelpResData_NRYRcleaned.txt`
   - `YelpChi/output_review_yelpResData_NRYRcleaned.txt`
   - `YelpChi/output_meta_yelpHotelData_NRYRcleaned.txt`
   - `YelpChi/output_review_yelpHotelData_NRYRcleaned.txt`
   - `YelpNYC/metadata`
   - `YelpNYC/reviewContent`

**Steps:**
1. No GPU needed — any runtime (CPU) is fine.
2. Update the file paths in **cell 1** if your files are in a subfolder on Google Drive.
3. Run all cells in order.
4. `yelp_split_imbalanced.csv` is saved to your Google Drive automatically.
5. Download `yelp_split_imbalanced.csv` and place it at `data/yelp_split_imbalanced.csv` locally — the leakage check in `evaluate_yelp_focal.py` / `evaluate_yelp_ablation.py` reads it from there.
6. The focal / contrastive training notebooks load `yelp_split_imbalanced.csv` directly from Google Drive — no local copy needed for training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── UPDATE THESE PATHS if your files are inside a subfolder ──
YELPZIP_META    = '/content/drive/MyDrive/YelpZip/metadata'
YELPZIP_REVIEWS = '/content/drive/MyDrive/YelpZip/reviewContent'

YELPCHI_META_RES    = '/content/drive/MyDrive/YelpChi/output_meta_yelpResData_NRYRcleaned.txt'
YELPCHI_REVIEW_RES  = '/content/drive/MyDrive/YelpChi/output_review_yelpResData_NRYRcleaned.txt'
YELPCHI_META_HOT    = '/content/drive/MyDrive/YelpChi/output_meta_yelpHotelData_NRYRcleaned.txt'
YELPCHI_REVIEW_HOT  = '/content/drive/MyDrive/YelpChi/output_review_yelpHotelData_NRYRcleaned.txt'

YELPNYC_META    = '/content/drive/MyDrive/YelpNYC/metadata'
YELPNYC_REVIEWS = '/content/drive/MyDrive/YelpNYC/reviewContent'

# ── The balanced split already on your Drive (generated by prepare_yelp_split.ipynb) ──
BALANCED_CSV = '/content/drive/MyDrive/yelp_split.csv'

# Output — used by focal loss training notebooks
SPLIT_CSV = '/content/drive/MyDrive/yelp_split_imbalanced.csv'

In [ ]:
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
def load_yelpzip(meta_path, review_path):
    """YelpZip / YelpNYC.
    metadata     : user_id  prod_id  rating  label  date   (tab-separated)
    reviewContent: user_id  prod_id  date    text          (tab-separated)
    Returns parallel lists: texts, labels, ratings, dates.
    """
    texts, labels, ratings, dates = [], [], [], []
    with open(meta_path, encoding='utf-8', errors='ignore') as mf, \
         open(review_path, encoding='utf-8', errors='ignore') as rf:
        for meta_line, review_line in zip(mf, rf):
            m = meta_line.strip().split('\t')
            r = review_line.strip().split('\t')
            if len(m) < 5 or len(r) < 4:
                continue
            label_str = m[3]
            if label_str == '-1':
                label = 1
            elif label_str == '1':
                label = 0
            else:
                continue
            text = r[3].strip()
            if text:
                texts.append(text)
                labels.append(label)
                ratings.append(m[2])   # star rating
                dates.append(m[4])     # review date (YYYY-MM-DD)
    return texts, labels, ratings, dates


def load_yelpchi(meta_path, review_path):
    """YelpChi.
    metadata: date  reviewID  reviewerID  productID  label(N/Y)  ...  rating(last col)
    review  : one review per line (the full text).
    Returns parallel lists: texts, labels, ratings, dates.
    """
    texts, labels, ratings, dates = [], [], [], []
    with open(meta_path, encoding='utf-8', errors='ignore') as mf, \
         open(review_path, encoding='utf-8', errors='ignore') as rf:
        for meta_line, review_line in zip(mf, rf):
            parts = meta_line.strip().split()
            if len(parts) < 5:
                continue
            label_str = parts[4]
            if label_str == 'Y':
                label = 1
            elif label_str == 'N':
                label = 0
            else:
                continue
            text = review_line.strip()
            if text:
                texts.append(text)
                labels.append(label)
                ratings.append(parts[8] if len(parts) > 8 else '')  # rating (last col)
                dates.append(parts[0])                              # review date (M/D/YYYY)
    return texts, labels, ratings, dates

In [ ]:
import re

def normalize_text(t):
    """Lowercase + collapse whitespace — for grouping/dedup only, not saved to CSV."""
    return re.sub(r'\s+', ' ', t.strip().lower())

print('Loading YelpZip ...')
zip_texts, zip_labels, zip_ratings, zip_dates = load_yelpzip(YELPZIP_META, YELPZIP_REVIEWS)
print(f'  {len(zip_texts):,} reviews')

print('Loading YelpChi (restaurants) ...')
chi_res_texts, chi_res_labels, chi_res_ratings, chi_res_dates = load_yelpchi(YELPCHI_META_RES, YELPCHI_REVIEW_RES)
print(f'  {len(chi_res_texts):,} reviews')

print('Loading YelpChi (hotels) ...')
chi_hot_texts, chi_hot_labels, chi_hot_ratings, chi_hot_dates = load_yelpchi(YELPCHI_META_HOT, YELPCHI_REVIEW_HOT)
print(f'  {len(chi_hot_texts):,} reviews')

print('Loading YelpNYC ...')
nyc_texts, nyc_labels, nyc_ratings, nyc_dates = load_yelpzip(YELPNYC_META, YELPNYC_REVIEWS)
print(f'  {len(nyc_texts):,} reviews')

all_texts   = zip_texts   + chi_res_texts   + chi_hot_texts   + nyc_texts
all_labels  = zip_labels  + chi_res_labels  + chi_hot_labels  + nyc_labels
all_ratings = zip_ratings + chi_res_ratings + chi_hot_ratings + nyc_ratings
all_dates   = zip_dates   + chi_res_dates   + chi_hot_dates   + nyc_dates

# Shuffle (keep text / label / rating / date aligned)
combined = list(zip(all_texts, all_labels, all_ratings, all_dates))
random.seed(SEED)
random.shuffle(combined)
all_texts, all_labels, all_ratings, all_dates = zip(*combined)
all_texts   = list(all_texts)
all_labels  = list(all_labels)
all_ratings = list(all_ratings)
all_dates   = list(all_dates)

n_fake    = sum(all_labels)
n_genuine = len(all_labels) - n_fake
print(f'\nTotal loaded, before dedup: {len(all_texts):,} reviews')
print(f'  Fake   : {n_fake:,}  ({n_fake/len(all_labels):.1%})')
print(f'  Genuine: {n_genuine:,}  ({n_genuine/len(all_labels):.1%})')

In [ ]:
# ── Step 0: build DataFrame and remove collection-duplicate reviews ───────────
# Same dedup rule as prepare_yelp_split.ipynb (keeps the two notebooks consistent).
# The SAME physical review appears in multiple source files (YelpNYC ⊂ YelpZip);
# those copies share identical (normalized text, rating, date) and are dropped to
# a single copy. "Natural" duplicates — different reviewers writing the same short
# text (e.g. "good") — differ in rating/date and are PRESERVED.
df = pd.DataFrame({
    'text':   all_texts,
    'label':  all_labels,
    'rating': all_ratings,
    'date':   all_dates,
})
df['text_norm'] = df['text'].apply(normalize_text)

before = len(df)
df = df.drop_duplicates(subset=['text_norm', 'rating', 'date'], keep='first').reset_index(drop=True)
removed = before - len(df)
print(f"Collection-duplicate removal: {before:,} → {len(df):,} rows  ({removed:,} removed, {removed/before:.1%})")
print(f"  Remaining fake  : {int(df['label'].sum()):,}  ({df['label'].mean():.1%})")
print(f"  Remaining total : {len(df):,}")
print(f"  Recommended FOCAL_ALPHA = {1 - df['label'].mean():.3f}  (genuine fraction; upweights minority fake class)")

# ── Step 1: load balanced split and extract pinned val/test text groups ───────
print('\nLoading balanced split to pin val/test text groups ...')
bal_df = pd.read_csv(BALANCED_CSV)
bal_df['text_norm'] = bal_df['text'].apply(normalize_text)

pinned_val_norms  = set(bal_df[bal_df['split'] == 'val' ]['text_norm'])
pinned_test_norms = set(bal_df[bal_df['split'] == 'test']['text_norm'])

overlap_pins = pinned_val_norms & pinned_test_norms
if overlap_pins:
    raise ValueError(
        f"{len(overlap_pins):,} normalized text(s) appear in both val and test "
        f"in yelp_split.csv. The balanced split is inconsistent — "
        f"regenerate yelp_split.csv before continuing."
    )

print(f'  Pinned val  groups : {len(pinned_val_norms):,}')
print(f'  Pinned test groups : {len(pinned_test_norms):,}')
print('  No overlap between pinned val and test — OK')

# ── Step 2: separate forced from remaining rows ───────────────────────────────
forced_mask  = df['text_norm'].isin(pinned_val_norms | pinned_test_norms)
forced_df    = df[forced_mask].copy()
remaining_df = df[~forced_mask].copy()

forced_df['split'] = forced_df['text_norm'].apply(
    lambda n: 'val' if n in pinned_val_norms else 'test'
)

pinned_val_rows  = int((forced_df['split'] == 'val').sum())
pinned_test_rows = int((forced_df['split'] == 'test').sum())
remaining_total  = len(remaining_df)

print(f'\nForced val  rows : {pinned_val_rows:,}')
print(f'Forced test rows : {pinned_test_rows:,}')
print(f'Remaining pool   : {remaining_total:,}')

# ── Step 3: compute how many extra rows each split needs ──────────────────────
total_rows        = len(df)
target_val_rows   = round(total_rows * 0.10)
target_test_rows  = round(total_rows * 0.10)
extra_val_needed  = max(0, target_val_rows  - pinned_val_rows)
extra_test_needed = max(0, target_test_rows - pinned_test_rows)

val_frac_rem  = extra_val_needed  / remaining_total
test_frac_rem = extra_test_needed / remaining_total

print(f'\nTarget val  : {target_val_rows:,}  ({pinned_val_rows:,} pinned + {extra_val_needed:,} extra from remaining)')
print(f'Target test : {target_test_rows:,}  ({pinned_test_rows:,} pinned + {extra_test_needed:,} extra from remaining)')
print(f'Target train: {total_rows - target_val_rows - target_test_rows:,}')
print(f'\nRemaining pool fractions — val: {val_frac_rem:.4f}  test: {test_frac_rem:.4f}  train: {1-val_frac_rem-test_frac_rem:.4f}')

# ── Step 4: grouped split of remaining pool using computed proportions ─────────
label_diversity = remaining_df.groupby('text_norm')['label'].nunique()
n_unique        = len(label_diversity)
n_mixed         = (label_diversity > 1).sum()
print(f'\nRemaining unique text groups : {n_unique:,}')
if n_mixed > 0:
    print(f'Mixed-label groups           : {n_mixed:,} — dominant label used')

text_label_df = (remaining_df.groupby('text_norm')['label']
                 .agg(lambda x: x.mode()[0])
                 .reset_index()
                 .rename(columns={'label': 'dominant_label'}))

tv_norms, te_norms, tv_labels, te_labels = train_test_split(
    text_label_df['text_norm'].values,
    text_label_df['dominant_label'].values,
    test_size=test_frac_rem, random_state=SEED,
    stratify=text_label_df['dominant_label'].values,
)
val_of_tv = val_frac_rem / (1.0 - test_frac_rem)
tr_norms, va_norms, tr_labels, va_labels = train_test_split(
    tv_norms, tv_labels,
    test_size=val_of_tv, random_state=SEED, stratify=tv_labels,
)

norm_to_split = {}
for t in tr_norms: norm_to_split[t] = 'train'
for t in va_norms: norm_to_split[t] = 'val'
for t in te_norms: norm_to_split[t] = 'test'
remaining_df['split'] = remaining_df['text_norm'].map(norm_to_split)

# ── Step 5: merge and report ──────────────────────────────────────────────────
df_final = pd.concat([forced_df, remaining_df], ignore_index=True)

print(f'\n{"Split":<8} {"Rows":>10}  {"Genuine":>10}  {"Fake":>9}  {"Fake%":>6}  {"% of total":>10}')
print('-' * 62)
for s in ('train', 'val', 'test'):
    sub     = df_final[df_final['split'] == s]
    n_g     = (sub['label'] == 0).sum()
    n_f     = (sub['label'] == 1).sum()
    print(f'{s:<8} {len(sub):>10,}  {n_g:>10,}  {n_f:>9,}  {n_f/len(sub):>6.1%}  {len(sub)/total_rows:>10.1%}')

df_final[['text', 'label', 'split']].to_csv(SPLIT_CSV, index=False)
print(f'\nSaved: {SPLIT_CSV}  ({len(df_final):,} rows)')

In [ ]:
print('=== Leakage Verification ===')
print()

train_norms = set(df_final[df_final['split'] == 'train']['text_norm'])
val_norms   = set(df_final[df_final['split'] == 'val'  ]['text_norm'])
test_norms  = set(df_final[df_final['split'] == 'test' ]['text_norm'])

tv_overlap = len(train_norms & val_norms)
tt_overlap = len(train_norms & test_norms)
vt_overlap = len(val_norms   & test_norms)

print('Unique normalized text groups per split:')
print(f'  Train : {len(train_norms):,}')
print(f'  Val   : {len(val_norms):,}')
print(f'  Test  : {len(test_norms):,}')
print()
print('Cross-split text-group overlaps (must all be 0):')
print(f'  Train ∩ Val  : {tv_overlap:,}')
print(f'  Train ∩ Test : {tt_overlap:,}')
print(f'  Val  ∩ Test  : {vt_overlap:,}')
print()

# The critical check: no yelp_split.csv test/val text may appear in focal training.
# evaluate_yelp_focal.py / evaluate_yelp_ablation.py evaluate all models on the
# yelp_split.csv test set, so any of those texts in focal train would be direct leakage.
leaked_test_in_train = pinned_test_norms & train_norms
leaked_val_in_train  = pinned_val_norms  & train_norms
print(f'Balanced test texts found in focal train : {len(leaked_test_in_train):,}  (must be 0)')
print(f'Balanced val  texts found in focal train : {len(leaked_val_in_train):,}  (must be 0)')
print()

# Informational: both notebooks use the same (text, rating, date) dedup, so every
# pinned group should still exist somewhere in the imbalanced dataset → expect ~0.
missing_from_test = pinned_test_norms - test_norms - train_norms - val_norms
missing_from_val  = pinned_val_norms  - val_norms  - train_norms - test_norms
print(f'Pinned test groups absent from imbalanced dataset : {len(missing_from_test):,}  (should be ~0)')
print(f'Pinned val  groups absent from imbalanced dataset : {len(missing_from_val):,}  (should be ~0)')
print()

if (
    tv_overlap == 0 and
    tt_overlap == 0 and
    vt_overlap == 0 and
    len(leaked_test_in_train) == 0 and
    len(leaked_val_in_train) == 0
):
    print('PASS — No leakage. evaluate_yelp_focal.py / ablation comparison is valid.')
else:
    raise ValueError(
        "Leakage verification failed — do not proceed to training.\n"
        f"  Train ∩ Val  : {tv_overlap}\n"
        f"  Train ∩ Test : {tt_overlap}\n"
        f"  Val  ∩ Test  : {vt_overlap}\n"
        f"  Balanced test texts in focal train : {len(leaked_test_in_train)}\n"
        f"  Balanced val  texts in focal train : {len(leaked_val_in_train)}"
    )

## Next steps
`yelp_split_imbalanced.csv` is saved to your Google Drive. Run the focal loss training notebooks:
- `train_yelp_lstm_focal.ipynb`
- `train_yelp_distilbert_focal.ipynb`

The baseline notebooks (`train_yelp_sklearn.ipynb`, `train_yelp_lstm.ipynb`, `train_yelp_distilbert.ipynb`) continue to use the balanced `yelp_split.csv` — do not change them.